In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Konfigurasi manajemen noise dengan Estimator

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan menggunakan versi ini atau yang lebih baru.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Ada beberapa cara untuk mengelola noise, biasanya dengan menggunakan berbagai teknik mitigasi error dan penekanan error untuk menghindari error sebelum terjadi. Teknik-teknik ini biasanya menyebabkan overhead pra-pemrosesan. Oleh karena itu, penting untuk mencapai keseimbangan antara menyempurnakan hasilmu dan memastikan jobmu selesai dalam waktu yang wajar.

Estimator mendukung teknik manajemen noise berikut. Lihat [Teknik mitigasi dan penekanan error](error-mitigation-and-suppression-techniques) untuk penjelasan masing-masing. Lihat bagian [Pengaturan error kustom](#advanced-error) untuk instruksi mengaktifkan teknik-teknik ini.

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Level resilience
`resilience_level` menentukan seberapa besar ketahanan yang dibangun terhadap error. Level yang lebih tinggi menghasilkan hasil yang lebih akurat, dengan biaya waktu pemrosesan yang lebih lama. Level resilience dapat digunakan untuk mengonfigurasi trade-off biaya/akurasi saat menerapkan manajemen noise ke kueri primitifmu. Manajemen noise mengurangi error (bias) dalam hasil dengan memproses output dari kumpulan, atau ansambel, Circuit yang saling berkaitan. Tingkat pengurangan error tergantung pada metode yang diterapkan. Level resilience mengabstraksi pilihan metode manajemen noise yang detail agar pengguna dapat mempertimbangkan trade-off biaya/akurasi yang sesuai untuk aplikasi mereka.

Dengan demikian, setiap level sesuai dengan satu metode atau lebih dengan level overhead sampling kuantum yang meningkat agar kamu bisa bereksperimen dengan trade-off waktu-akurasi yang berbeda. Tabel berikut menunjukkan level dan metode yang sesuai yang tersedia untuk masing-masing primitif.

<span id="resilience-table"></span>

| Level Resilience | Deskripsi                                                                                            | Teknik                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Tanpa mitigasi                                                                                         | Tidak ada                                                                      |
| 1 [Default]      | Biaya mitigasi minimal: Memitigasi error yang terkait dengan error readout                               | Pengukuran twirling [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex)             |
| 2                | Biaya mitigasi sedang. Biasanya mengurangi bias dalam estimator, tapi tidak dijamin zero-bias. | Level 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) dan gate twirling

> **Info:** Level resilience saat ini masih dalam beta sehingga overhead sampling dan kualitas solusi akan bervariasi dari satu Circuit ke Circuit lainnya. Fitur baru, opsi lanjutan, dan tool manajemen akan dirilis secara bertahap. Metode manajemen noise tertentu tidak dijamin diterapkan pada setiap level resilience.

### Konfigurasi Estimator dengan level resilience
Kamu bisa menggunakan level resilience untuk menentukan teknik manajemen noise, atau kamu bisa mengatur teknik kustom secara individual seperti yang dijelaskan di [Pengaturan error kustom](#advanced-error).

> **Note:** Opsi apa pun yang kamu tentukan secara manual selain level resilience diterapkan di samping set opsi dasar yang ditentukan oleh level resilience. Oleh karena itu, pada prinsipnya, kamu bisa mengatur level resilience ke 1, tapi kemudian menonaktifkan mitigasi pengukuran, meskipun ini tidak disarankan.
> 
> Misalnya, mengatur level resilience ke 0 menonaktifkan `zne_mitigation`, tapi `estimator.options.resilience.zne_mitigation = True` menimpa nilai tersebut.

### Contoh
Kode berikut mengaktifkan ZNE, TREX, dan gate twirling dengan mengatur `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Pengaturan manajemen noise kustom
Kamu bisa mengaktifkan dan menonaktifkan metode manajemen noise individual menggunakan [opsi Estimator](/guides/estimator-options).

> **Note:** Tidak semua opsi bisa digunakan bersama pada semua jenis Circuit. Lihat [tabel kompatibilitas fitur](estimator-options#options-compatibility-table) untuk detailnya.

### Contoh

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>